In [0]:
%sql
create or replace table demo_modelling_demo_catalog.gold_snowflake.fact_sales AS
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.payment_id,
    o.order_total,
    oi.order_item_id,
    oi.product_id,
    oi.quantity,
    oi.unit_price,
    oi.discount_amount,
    oi.line_total
FROM demo_modelling_demo_catalog.silver.orders o
INNER JOIN demo_modelling_demo_catalog.silver.order_items oi
    ON o.order_id = oi.order_id;

In [0]:
%sql
table demo_modelling_demo_catalog.gold_snowflake.fact_sales

In [0]:
%sql
create or replace table demo_modelling_demo_catalog.gold_snowflake.dim_geography as
select
ci.city_id,
ci.city_name,
s.state_name,
c.country_name,
c.region
from demo_modelling_demo_catalog.silver.cities ci join  demo_modelling_demo_catalog.silver.states s
on ci.state_id = s.state_id
join demo_modelling_demo_catalog.silver.countries c
on s.country_id = c.country_id;


In [0]:
%sql
table demo_modelling_demo_catalog.gold_snowflake.dim_geography;

In [0]:
%sql
create or replace view demo_modelling_demo_catalog.gold_snowflake.dim_products as 
select * 
from 
demo_modelling_demo_catalog.silver.products;

In [0]:
%sql
create or replace view demo_modelling_demo_catalog.gold_snowflake.dim_customers as 
select * 
from 
demo_modelling_demo_catalog.silver.customers;

In [0]:
%sql
create or replace view demo_modelling_demo_catalog.gold_snowflake.dim_payments as 
select * 
from 
demo_modelling_demo_catalog.silver.payments;

In [0]:
%sql
create or replace view demo_modelling_demo_catalog.gold_snowflake.dim_brands as 
select * 
from 
demo_modelling_demo_catalog.silver.brands;

In [0]:
%sql
create or replace view demo_modelling_demo_catalog.gold_snowflake.dim_subcategories as 
select * 
from 
demo_modelling_demo_catalog.silver.subcategories;

In [0]:
%sql
create or replace view demo_modelling_demo_catalog.gold_snowflake.dim_categories as 
select * 
from 
demo_modelling_demo_catalog.silver.categories;

In [0]:
%sql
CREATE OR REPLACE TABLE demo_modelling_demo_catalog.gold_snowflake.dim_date AS

WITH date_range AS (
    SELECT EXPLODE(
        SEQUENCE(
            DATE('2023-01-01'),
            DATE('2036-12-31'),
            INTERVAL 1 DAY
        )
    ) AS date_day
)

SELECT
    CAST(DATE_FORMAT(date_day, 'yyyyMMdd') AS INT) AS date_key,
    date_day AS full_date,

    -- Day attributes
    DAY(date_day) AS day_of_month,
    DAYOFWEEK(date_day) AS day_of_week,
    DATE_FORMAT(date_day, 'EEEE') AS day_name,
    DAYOFYEAR(date_day) AS day_of_year,

    -- Month attributes
    MONTH(date_day) AS month_number,
    DATE_FORMAT(date_day, 'MMMM') AS month_name,
    DATE_FORMAT(date_day, 'MMM') AS month_short_name,

    -- Quarter attributes
    QUARTER(date_day) AS calendar_quarter,
    CONCAT('Q', QUARTER(date_day)) AS quarter_name,

    -- Calendar year
    YEAR(date_day) AS calendar_year,

    -- Fiscal year: April to March
    CASE
        WHEN MONTH(date_day) >= 4
        THEN YEAR(date_day)
        ELSE YEAR(date_day) - 1
    END AS fiscal_year_start,

    CONCAT(
        'FY ',
        CASE
            WHEN MONTH(date_day) >= 4
            THEN YEAR(date_day)
            ELSE YEAR(date_day) - 1
        END,
        '-',
        RIGHT(
            CAST(
                CASE
                    WHEN MONTH(date_day) >= 4
                    THEN YEAR(date_day) + 1
                    ELSE YEAR(date_day)
                END AS STRING
            ),
            2
        )
    ) AS fiscal_year,

    CASE
        WHEN MONTH(date_day) >= 4
        THEN MONTH(date_day) - 3
        ELSE MONTH(date_day) + 9
    END AS fiscal_month_number,

    CASE
        WHEN MONTH(date_day) BETWEEN 4 AND 6 THEN 'Q1'
        WHEN MONTH(date_day) BETWEEN 7 AND 9 THEN 'Q2'
        WHEN MONTH(date_day) BETWEEN 10 AND 12 THEN 'Q3'
        ELSE 'Q4'
    END AS fiscal_quarter

FROM date_range;


In [0]:
%sql
table demo_modelling_demo_catalog.gold_snowflake.dim_date;


In [0]:
%sql
SELECT
    dg.country_name,
    SUM(fs.line_total) AS total_sales
FROM demo_modelling_demo_catalog.gold_snowflake.fact_sales fs
JOIN demo_modelling_demo_catalog.gold_snowflake.dim_customers dc
    ON fs.customer_id = dc.customer_id
JOIN demo_modelling_demo_catalog.gold_snowflake.dim_geography dg
    ON dc.city_id = dg.city_id
GROUP BY dg.country_name
ORDER BY total_sales DESC;